In [ ]:
# version 1.0

In [96]:
import pandas as pd

# root_path: The directory where the dataset is downloaded or hosted.
root_path = "https://huggingface.co/datasets/m-sakka/agripotential/resolve/main/"
path = root_path + "metadata.csv"
metadata = pd.read_csv(path)
metadata.head()

,filename,day,month,year
0,T31TEJ_2017_01_03.tif,3,1,2017
1,T31TEJ_2017_03_29.tif,29,3,2017
2,T31TEJ_2017_06_17.tif,17,6,2017
3,T31TEJ_2017_07_07.tif,7,7,2017
4,T31TEJ_2017_07_17.tif,17,7,2017


In [97]:
from rasterio import rasterio
from rasterio.windows import Window

train_subset_path = root_path + "train.csv"

train_df = pd.read_csv(train_subset_path)
train_df.head()



date1 = metadata.iloc[0]


print(date1)


filename    T31TEJ_2017_01_03.tif
day                             3
month                           1
year                         2017
Name: 0, dtype: object


In [98]:
# to start, will focus on >T31TEJ_2017_01_03.tif,03,01,2017<

In [99]:
''' sources to read:
Vision Transformer (ViT)
Dosovitskiy et al., 2020

Attention Is All You Need
Vaswani et al., 2017

Spectral–Spatial Transformer for Hyperspectral Image Classification
He et al., 2021

SpectralFormer: Rethinking Hyperspectral Image Classification with Transformers
Zhang et al., 2021
'''

' sources to read:\nVision Transformer (ViT)\nDosovitskiy et al., 2020\n\nAttention Is All You Need\nVaswani et al., 2017\n\nSpectral–Spatial Transformer for Hyperspectral Image Classification\nHe et al., 2021\n\nSpectralFormer: Rethinking Hyperspectral Image Classification with Transformers\nZhang et al., 2021\n'

In [100]:
date1_data = rasterio.open(root_path+date1["filename"])
# date1_data.meta

In [101]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [102]:


x = []

for index in range(3):
  patch_row = train_df.iloc[index]["row"]
  patch_col = train_df.iloc[index]["col"]
  patch_size = train_df.iloc[index]["patch_size"]
  patch_id = train_df.iloc[index]["patch_id"]
  image = date1_data.read(window=Window(patch_col, patch_row, patch_size, patch_size))
  x.append(torch.from_numpy(image))

x = torch.stack(x, dim=0)
x = x.to(device)



In [103]:
class SpectralViTPixel(nn.Module):
    def __init__(self, num_bands, num_classes, d_model=128, depth=4, nhead=4):
        super().__init__()
        self.num_bands = num_bands

        # 1. Embed each spectral band
        self.band_embed = nn.Linear(1, d_model)

        # 2. Positional encoding across wavelengths
        self.pos_embed = nn.Parameter(torch.zeros(1, num_bands, d_model))

        # 3. Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # 4. Classification head
        self.cls_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        B, C, H, W = x.shape

        # Flatten to pixels: (B*H*W, C, 1)
        x = x.permute(0, 2, 3, 1).reshape(-1, C, 1)

        # Band embedding
        x = self.band_embed(x)  # (Npix, C, d_model)

        # Add spectral positional encoding
        x = x + self.pos_embed

        # Transformer encoder
        x = self.encoder(x)  # (Npix, C, d_model)

        # Mean pooling over spectral dimension
        x = x.mean(dim=1)  # (Npix, d_model)

        # Class logits
        logits = self.cls_head(x)  # (Npix, num_classes)

        # Reshape back to image
        return logits.view(B, H, W, -1).permute(0, 3, 1, 2)

In [104]:
model = SpectralViTPixel(num_bands=10,num_classes=5).to(device)

In [105]:
x = x.to(torch.float32)
logits = model.forward(x)

In [106]:
viticulture_label_path = root_path + "viticulture.tif"
viticulture_label_data = rasterio.open(viticulture_label_path)

Y = []

for index in range(3):
  patch_row = train_df.iloc[index]["row"]
  patch_col = train_df.iloc[index]["col"]
  patch_size = train_df.iloc[index]["patch_size"]
  patch_id = train_df.iloc[index]["patch_id"]
  image = viticulture_label_data.read(window=Window(patch_col, patch_row, patch_size, patch_size))
  Y.append(torch.from_numpy(image))

Y = torch.stack(Y, dim=0)
Y = Y.to(device)

labels = Y.reshape(-1)

In [107]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()
total_loss = 0
total_correct = 0
total_pixels = 0


In [108]:
print(logits.shape)

torch.Size([3, 5, 128, 128])


In [109]:
B, K, H, W = logits.shape
logits = logits.permute(0, 2, 3, 1).reshape(-1, K)
mask = labels != 255
logits = logits[mask]
labels = labels[mask]

loss = criterion(logits, labels).to(device)

optimizer.zero_grad()
loss.backward()
optimizer.step()

total_loss += loss.item() * labels.numel()

# Accuracy
preds = logits.argmax(dim=1)
total_correct += (preds == labels).sum().item()
total_pixels += labels.numel()


In [110]:
avg_loss = total_loss / total_pixels
accuracy = total_correct / total_pixels
print(f"avg_loss: {avg_loss}; accuracy: {accuracy}")

avg_loss: 1.5791703462600708; accuracy: 0.018147786458333332
